# Антиспам NLP

Запускать из корня проекта после установки requirements.txt. Архив преподавателя нужен только для итоговой проверки. Не открывать письма и вложения вручную.

In [ ]:
from pathlib import Path
import json
from app.text import read_archive


## 1. Сбор и обработка

SpamAssassin и LingSpam. Извлекаем только тему и текст, удаляем повторы. Не включаем антиспам-заголовки в признаки.

In [ ]:
%run download_data.py

## 2. Обучение и проверка

Два классификатора, 5-fold CV, отдельная валидация для выбора порога. Следующая ячейка заново выполняет обучение. Укажите путь к тестовому архиву.

In [ ]:
test_path = Path('test_task_emails.tar.xz')
assert test_path.exists(), 'Укажите путь к архиву преподавателя'
from train import main
main(test_path)

In [ ]:
metrics = json.loads(Path('reports/metrics.json').read_text())
metrics['experiments'], metrics['validation'], metrics['test']

## 3. Сохранённый результат выполненного эксперимента

Precision 99.34%, Recall 90.98%, F1 94.98%. TP=908, FP=6, FN=90, TN=594. Эти числа взяты из отчёта, а не из исполнения ячеек этого файла.

In [ ]:
import joblib
bundle = joblib.load('artifacts/model.joblib')
texts = ['Project meeting on Tuesday. Please review our report.', 'Win free cash now! Claim your lottery prize!']
bundle['pipeline'].predict_proba(texts), bundle['threshold']

## 4. Дополнительная модель

Категории обучены на искусственных данных. Оценка на реальной почте отсутствует.

In [ ]:
%run train_categories.py

## 5. Сервис и выводы

Запуск: `python -m uvicorn app.main:app --port 8000`.

Самостоятельно ответьте: чем FP отличается от FN? Почему порог выбирается на validation? Какие ограничения у исторических англоязычных данных?